# Classifying Illicit Bitcoin Transactions on the Elliptic Data Set
## An End-to-End Comparison of Tabular, Graph-Engineered, and Graph Neural Network Models with a Time-Based Distribution Shift Diagnosis

**Author:** Isaak Alemu  
**Context:** INSA Ethiopia Summer Camp — Emerging Track

---

### Project Overview & Roadmap
This notebook provides a complete, self-contained implementation of Bitcoin illicit transaction detection on the **Elliptic Data Set** (~203,769 transactions, 234,355 directed edges, 165 features across 49 time steps).

The project is structured into distinct, modular stages:
1. **Unified Setup & Data Ingestion**: Consolidated dependencies, reproducible seeds, and environment handling.
2. **Exploratory Data Analysis & Tabular Baseline (Random Forest)**: Evaluation on 165 tabular features using a strict temporal train/test split (steps 1–34 train, 35–49 test).
3. **Hand-Built Graph Features — Thin Version (Random Forest)**: Adding in/out degree, PageRank, and 1-hop neighbor means (explicitly noting Random Forest usage).
4. **Hand-Built Graph Features — Full Version (XGBoost)**: Expanding to undirected clustering coefficient, 2-hop reach, and full 165-feature neighbor aggregations with XGBoost (noting the model switch and motivation).
5. **Consistent XGBoost Comparison (Apples-to-Apples Evaluation)**: Comparing Tabular, Thin Graph, and Full Graph under an identical classifier, plus **Confusion Matrices**, **Split-Boundary Ablation**, and **SHAP Feature Explanations**.
6. **Distribution Shift Diagnosis**: Statistical quantification (KS tests, confidence histograms, feature shift analysis) explaining the abrupt step-43 performance collapse.
7. **Graph Neural Network (GCN with PyTorch Geometric — All 165 Features)**: End-to-end message passing over the transaction graph.
8. **Additional Exploration (GCN with 94 Local Features Only)**: Evaluating GCN behavior when pre-aggregated neighbor features are removed.
9. **Literature Comparison (Weber et al. 2019)**: Benchmarking against the original KDD 2019 publication.
10. **Discussion & Production Deployment Strategies**: Architectural considerations for real-world anti-money laundering (AML) systems.


---
# 1. Setup & Environment
Consolidated package installation, universal data loading (supporting local execution, Google Drive, or Kaggle download), and global configuration.


In [ ]:
# 1. Setup & Package Installations (Automated for Google Colab / Local)
import sys
import os

# Automatically install required packages in Colab
if 'google.colab' in sys.modules:
    get_ipython().system("pip install -q scikit-learn networkx xgboost matplotlib pandas scipy torch torch_geometric shap kaggle")

import json
import warnings
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import scipy.stats as stats
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
import xgboost as xgb

# Set random seeds for strict reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
print("Environment and core libraries initialized successfully.")


### Data Ingestion Configuration
The dataset consists of three CSV files:
- `elliptic_txs_features.csv`: 166 anonymized features (col 0: `txId`, col 1: `time_step`, cols 2..166: `feat_1` to `feat_165`).
- `elliptic_txs_classes.csv`: Transaction class labels (`1` = illicit, `2` = licit, `unknown`).
- `elliptic_txs_edgelist.csv`: Directed graph transaction flows (`txId1` $\to$ `txId2`).

Configure the data path below. If running in Google Colab, you can mount Google Drive or download directly via Kaggle API.


In [ ]:
# 2. Automated Dataset Ingestion & Path Resolution
# Checks local folders, Colab/Drive paths, and auto-downloads if missing.

candidate_paths = [
    "../data/elliptic_bitcoin_dataset",
    "./data/elliptic_bitcoin_dataset",
    "../data",
    "./data",
    ".",
    "/content/data/elliptic_bitcoin_dataset",
    "/content/data",
    "/content/drive/MyDrive/elliptic_project/elliptic_bitcoin_dataset",
    "/content/drive/MyDrive/elliptic_project"
]

DATA_DIR = None
for p in candidate_paths:
    if os.path.exists(os.path.join(p, "elliptic_txs_features.csv")) and \
       os.path.exists(os.path.join(p, "elliptic_txs_classes.csv")) and \
       os.path.exists(os.path.join(p, "elliptic_txs_edgelist.csv")):
        DATA_DIR = p
        break

# (Optional) Kaggle API token if you prefer downloading directly from Kaggle
KAGGLE_TOKEN = "PASTE_YOUR_TOKEN_HERE"

if DATA_DIR is None:
    # Determine target download directory
    if 'google.colab' in sys.modules or os.path.exists('/content'):
        target_dir = "/content/data/elliptic_bitcoin_dataset"
    else:
        target_dir = "../data/elliptic_bitcoin_dataset"
    
    os.makedirs(target_dir, exist_ok=True)
    
    if KAGGLE_TOKEN != "PASTE_YOUR_TOKEN_HERE" and len(KAGGLE_TOKEN.strip()) > 0:
        print("Using Kaggle API token to download dataset...")
        os.makedirs('/root/.kaggle', exist_ok=True)
        with open('/root/.kaggle/access_token', 'w') as f:
            f.write(KAGGLE_TOKEN.strip())
        os.chmod('/root/.kaggle/access_token', 0o600)
        get_ipython().system("pip install -q kaggle")
        get_ipython().system("kaggle datasets download -d ellipticco/elliptic-data-set -p /content/data --unzip")
        DATA_DIR = "/content/data/elliptic_bitcoin_dataset" if os.path.exists("/content/data/elliptic_bitcoin_dataset/elliptic_txs_features.csv") else "/content/data"
    else:
        print(f"Dataset not found locally. Downloading directly to '{target_dir}' (no API key required)...")
        base_url = "https://huggingface.co/datasets/yhoma/elliptic-bitcoin-dataset/resolve/main"
        required_files = [
            "elliptic_txs_classes.csv",
            "elliptic_txs_edgelist.csv",
            "elliptic_txs_features.csv"
        ]
        for fname in required_files:
            dest_file = os.path.join(target_dir, fname)
            if not os.path.exists(dest_file):
                print(f"  Downloading {fname}...")
                urllib.request.urlretrieve(f"{base_url}/{fname}", dest_file)
                print(f"  Saved {fname} ({os.path.getsize(dest_file):,} bytes).")
        DATA_DIR = target_dir

print(f"\nData path successfully resolved: using '{DATA_DIR}'")


In [ ]:
# Load raw CSV files
feat_cols = ["txId", "time_step"] + [f"feat_{i}" for i in range(1, 166)]
features = pd.read_csv(os.path.join(DATA_DIR, "elliptic_txs_features.csv"), header=None, names=feat_cols)

classes = pd.read_csv(os.path.join(DATA_DIR, "elliptic_txs_classes.csv"))
classes.columns = ["txId", "class"]

edges = pd.read_csv(os.path.join(DATA_DIR, "elliptic_txs_edgelist.csv"))
edges.columns = ["txId1", "txId2"]

print(f"Features shape: {features.shape}")
print(f"Classes shape:  {classes.shape}")
print(f"Edges shape:    {edges.shape}")

# Merge features and classes into master DataFrame
df = features.merge(classes, on="txId", how="left")

# Label mapping: '1' -> illicit (1), '2' -> licit (0), 'unknown' -> -1 / excluded
df["label"] = df["class"].map({"1": 1, "2": 0})
labeled_df = df[df["class"] != "unknown"].copy()
unknown_df = df[df["class"] == "unknown"].copy()

print(f"Total Transactions:   {len(df):,}")
print(f"Labeled Transactions: {len(labeled_df):,} ({len(labeled_df)/len(df):.1%})")
print(f"  - Licit (0):        {(labeled_df['label'] == 0).sum():,} ({(labeled_df['label'] == 0).mean():.1%})")
print(f"  - Illicit (1):      {(labeled_df['label'] == 1).sum():,} ({(labeled_df['label'] == 1).mean():.1%})")
print(f"Unknown Transactions: {len(unknown_df):,} ({len(unknown_df)/len(df):.1%})")


---
# 2. Exploratory Data Analysis & Tabular Baseline (Random Forest)

### Temporal Train/Test Split
In financial anti-money laundering (AML), transactions occur chronologically. A random train/test split would cause **temporal data leakage**, creating overly optimistic performance estimates. 

Following standard methodology (Weber et al., 2019):
- **Training Set**: Time steps 1 through 34 (~70% chronological window)
- **Test Set**: Time steps 35 through 49 (~30% future window)


In [ ]:
# EDA: Class balance over time
illicit_by_step = labeled_df[labeled_df["label"] == 1].groupby("time_step").size()
licit_by_step = labeled_df[labeled_df["label"] == 0].groupby("time_step").size()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(licit_by_step.index, licit_by_step.values, marker="o", color="#2b5c8f", label="Licit Transactions (Class 2 -> 0)")
ax.plot(illicit_by_step.index, illicit_by_step.values, marker="s", color="#d95f02", label="Illicit Transactions (Class 1 -> 1)")
ax.axvline(34.5, color="gray", linestyle="--", alpha=0.7, label="Train/Test Cutoff (Step 34)")
ax.set_xlabel("Time Step (each ~2 weeks)")
ax.set_ylabel("Transaction Count")
ax.set_title("Labeled Bitcoin Transactions per Time Step (Elliptic Dataset)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Construct the Temporal Train/Test Split
TRAIN_MAX_STEP = 34

train_df = labeled_df[labeled_df["time_step"] <= TRAIN_MAX_STEP].copy()
test_df  = labeled_df[labeled_df["time_step"] > TRAIN_MAX_STEP].copy()

feature_cols = [c for c in df.columns if c.startswith("feat_")]

X_train, y_train = train_df[feature_cols], train_df["label"]
X_test, y_test = test_df[feature_cols], test_df["label"]

print(f"Train Set: {X_train.shape[0]:,} samples | Illicit rate: {y_train.mean():.3%}")
print(f"Test Set:  {X_test.shape[0]:,} samples | Illicit rate: {y_test.mean():.3%}")


### Tabular Baseline Model: Random Forest
We train a Random Forest classifier with `class_weight='balanced'` on the 165 tabular features.


In [ ]:
# Train Random Forest Tabular Baseline
clf_rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced"
)
clf_rf.fit(X_train, y_train)

y_pred_rf = clf_rf.predict(X_test)
test_df["pred_rf"] = y_pred_rf
test_df["pred_proba_rf"] = clf_rf.predict_proba(X_test)[:, 1]

print("=== Tabular Baseline (Random Forest) Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=["licit", "illicit"], digits=4))

rf_tabular_overall_f1 = f1_score(y_test, y_pred_rf, pos_label=1)
print(f"Overall Illicit-Class F1: {rf_tabular_overall_f1:.4f}")


In [ ]:
# Calculate F1 per time step for Tabular Baseline
f1_per_step_rf = []
for step, group in test_df.groupby("time_step"):
    f1 = f1_score(group["label"], group["pred_rf"], pos_label=1, zero_division=0)
    f1_per_step_rf.append((step, f1))

steps_rf, f1s_rf = zip(*f1_per_step_rf)

plt.figure(figsize=(10, 4))
plt.plot(steps_rf, f1s_rf, marker="o", color="#2b5c8f", lw=2, label="Tabular Baseline (RF)")
plt.axvline(43, color="#d95f02", linestyle="--", alpha=0.7, label="Collapse Start (Step 43)")
plt.xlabel("Time Step")
plt.ylabel("Illicit-Class F1 Score")
plt.title("Tabular Baseline (Random Forest): Illicit F1 Over Time")
plt.legend()
plt.tight_layout()
plt.show()

rf_healthy_f1 = f1_score(test_df[test_df.time_step.between(35, 42)]["label"], test_df[test_df.time_step.between(35, 42)]["pred_rf"], pos_label=1)
rf_collapsed_f1 = f1_score(test_df[test_df.time_step >= 43]["label"], test_df[test_df.time_step >= 43]["pred_rf"], pos_label=1)

print(f"Healthy Period F1 (Steps 35-42):   {rf_healthy_f1:.4f}")
print(f"Collapsed Period F1 (Steps 43-49): {rf_collapsed_f1:.4f}")


---
# 3. Hand-Built Graph Features — Thin Version (Random Forest)

> [!NOTE]
> **Model Consistency Note**: This stage intentionally uses **Random Forest** (`clf_rf_thin`) to ensure a direct, single-variable comparison against the tabular baseline from Section 2.

We construct a directed NetworkX graph `G` from the transaction edgelist and compute:
1. **In-Degree & Out-Degree**: Directional transaction volume.
2. **PageRank**: Transaction centrality in the money flow network.
3. **1-Hop Neighborhood Aggregates**: Mean values of the first 10 local tabular features across predecessors and successors.


In [ ]:
# Build Directed Graph with NetworkX
G = nx.DiGraph()
G.add_nodes_from(df["txId"])
G.add_edges_from(edges[["txId1", "txId2"]].itertuples(index=False, name=None))

print(f"Graph Construction Complete: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} directed edges.")


In [ ]:
# Compute Degree & PageRank
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

print("Computing PageRank across 203k nodes...")
pagerank = nx.pagerank(G, alpha=0.85)
print("PageRank computation complete.")

graph_feat_thin = pd.DataFrame({
    "txId": list(G.nodes()),
    "in_degree": [in_degree[n] for n in G.nodes()],
    "out_degree": [out_degree[n] for n in G.nodes()],
    "pagerank": [pagerank[n] for n in G.nodes()],
})

# Compute 1-Hop Neighbor Means for 10 local features
agg_subset_cols = feature_cols[:10]
feat_lookup_10 = df.set_index("txId")[agg_subset_cols]

neighbor_means_10 = {}
for node in G.nodes():
    neighbors = list(G.predecessors(node)) + list(G.successors(node))
    if not neighbors:
        neighbor_means_10[node] = np.zeros(len(agg_subset_cols))
        continue
    neighbor_vals = feat_lookup_10.reindex(neighbors).values
    neighbor_means_10[node] = np.nanmean(neighbor_vals, axis=0)

neighbor_df_10 = pd.DataFrame({node: vals for node, vals in neighbor_means_10.items()}).T
neighbor_df_10.columns = [f"nbr_mean_{c}" for c in agg_subset_cols]
neighbor_df_10.index.name = "txId"
neighbor_df_10 = neighbor_df_10.reset_index()

# Merge into thin graph DataFrame
df_thin_graph = df.merge(graph_feat_thin, on="txId", how="left").merge(neighbor_df_10, on="txId", how="left")
thin_nbr_cols = [c for c in df_thin_graph.columns if c.startswith("nbr_mean_")]
df_thin_graph[thin_nbr_cols] = df_thin_graph[thin_nbr_cols].fillna(0)

print("Thin graph features merged successfully. Shape:", df_thin_graph.shape)


In [ ]:
# Train Random Forest on Tabular + Thin Graph Features
labeled_thin_df = df_thin_graph[df_thin_graph["class"] != "unknown"].copy()

train_thin = labeled_thin_df[labeled_thin_df["time_step"] <= TRAIN_MAX_STEP]
test_thin  = labeled_thin_df[labeled_thin_df["time_step"] > TRAIN_MAX_STEP].copy()

thin_feature_cols = feature_cols + ["in_degree", "out_degree", "pagerank"] + thin_nbr_cols

X_train_thin, y_train_thin = train_thin[thin_feature_cols], train_thin["label"]
X_test_thin, y_test_thin = test_thin[thin_feature_cols], test_thin["label"]

clf_rf_thin = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced"
)
clf_rf_thin.fit(X_train_thin, y_train_thin)

y_pred_rf_thin = clf_rf_thin.predict(X_test_thin)
test_thin["pred"] = y_pred_rf_thin

print("=== Tabular + Thin Graph (Random Forest) Classification Report ===")
print(classification_report(y_test_thin, y_pred_rf_thin, target_names=["licit", "illicit"], digits=4))

rf_thin_overall_f1 = f1_score(y_test_thin, y_pred_rf_thin, pos_label=1)
print(f"Overall Illicit-Class F1 (RF Thin Graph): {rf_thin_overall_f1:.4f}")


In [ ]:
# Feature Importance Analysis for Thin Graph RF
importances_thin = pd.Series(clf_rf_thin.feature_importances_, index=thin_feature_cols).sort_values(ascending=False)
thin_added_cols = set(["in_degree", "out_degree", "pagerank"] + thin_nbr_cols)

print("Top 15 Features Overall in Thin Graph Model:")
print(importances_thin.head(15))

print("\nRanks of Hand-Built Graph Features (out of %d total features):" % len(thin_feature_cols))
ranks_thin = importances_thin.reset_index()
ranks_thin.columns = ["feature", "importance"]
ranks_thin["rank"] = ranks_thin.index + 1
print(ranks_thin[ranks_thin["feature"].isin(thin_added_cols)])


---
# 4. Hand-Built Graph Features — Full Version (XGBoost)

> [!IMPORTANT]
> **Model Switch Rationale**: In this stage, we expand graph feature engineering significantly:
> - Undirected **Clustering Coefficient**
> - **2-Hop Reachable Node Count**
> - Full-width **1-Hop Neighbor Mean & Max** across all 165 tabular features (330 additional columns)
>
> Because these richer, high-dimensional features create complex non-linear interactions, we introduced **XGBoost** as a more powerful gradient-boosted tree learner. However, this conflates two factors: the new features and the new model architecture. This observation directly motivates the strict, apples-to-apples comparison in Section 5.


In [ ]:
# Compute Undirected Clustering Coefficient and 2-Hop Reach
print("Computing clustering coefficient (undirected view)...")
G_undirected = G.to_undirected()
clustering_coef = nx.clustering(G_undirected)

print("Computing 2-hop reach...")
two_hop_reach = {}
for node in G.nodes():
    one_hop = set(G.successors(node)) | set(G.predecessors(node))
    two_hop = set()
    for n in one_hop:
        two_hop |= set(G.successors(n)) | set(G.predecessors(n))
    two_hop_reach[node] = len(two_hop - one_hop - {node})

graph_feat_full = pd.DataFrame({
    "txId": list(G.nodes()),
    "in_degree": [in_degree[n] for n in G.nodes()],
    "out_degree": [out_degree[n] for n in G.nodes()],
    "pagerank": [pagerank[n] for n in G.nodes()],
    "clustering_coef": [clustering_coef[n] for n in G.nodes()],
    "two_hop_reach": [two_hop_reach[n] for n in G.nodes()],
})
print("Structural graph metrics computed.")


In [ ]:
# Full-width 1-Hop Neighbor Mean and Max across all 165 features
feat_lookup_all = df.set_index("txId")[feature_cols]
feat_matrix_all = feat_lookup_all.values
txid_to_idx = {tx: i for i, tx in enumerate(feat_lookup_all.index)}

node_list = list(G.nodes())
num_nodes = len(node_list)
num_feats = len(feature_cols)

nbr_mean_matrix = np.zeros((num_nodes, num_feats), dtype=np.float32)
nbr_max_matrix = np.zeros((num_nodes, num_feats), dtype=np.float32)

print("Aggregating mean and max neighbor statistics across all 165 features...")
for i, node in enumerate(node_list):
    nbrs = list(G.predecessors(node)) + list(G.successors(node))
    if not nbrs:
        continue
    idxs = [txid_to_idx[n] for n in nbrs if n in txid_to_idx]
    if not idxs:
        continue
    vals = feat_matrix_all[idxs]
    nbr_mean_matrix[i] = np.nanmean(vals, axis=0)
    nbr_max_matrix[i] = np.nanmax(vals, axis=0)

df_nbr_mean = pd.DataFrame(nbr_mean_matrix, columns=[f"nbr_mean_{c}" for c in feature_cols])
df_nbr_max = pd.DataFrame(nbr_max_matrix, columns=[f"nbr_max_{c}" for c in feature_cols])
df_nbr_mean.insert(0, "txId", node_list)
df_nbr_max.insert(0, "txId", node_list)

# Master Full Graph DataFrame
df_full_graph = (df
    .merge(graph_feat_full, on="txId", how="left")
    .merge(df_nbr_mean, on="txId", how="left")
    .merge(df_nbr_max, on="txId", how="left"))

full_nbr_cols = [c for c in df_full_graph.columns if c.startswith(("nbr_mean_", "nbr_max_"))]
df_full_graph[full_nbr_cols] = df_full_graph[full_nbr_cols].fillna(0)

full_graph_extra_cols = ["in_degree", "out_degree", "pagerank", "clustering_coef", "two_hop_reach"] + full_nbr_cols
full_feature_cols = feature_cols + full_graph_extra_cols

print(f"Full-Graph dataset assembled: {df_full_graph.shape[1]} total columns ({len(full_feature_cols)} feature columns).")


In [ ]:
# Train XGBoost on Tabular + Full Graph Features
labeled_full_df = df_full_graph[df_full_graph["class"] != "unknown"].copy()

train_full = labeled_full_df[labeled_full_df["time_step"] <= TRAIN_MAX_STEP]
test_full  = labeled_full_df[labeled_full_df["time_step"] > TRAIN_MAX_STEP].copy()

X_train_full, y_train_full = train_full[full_feature_cols], train_full["label"]
X_test_full, y_test_full = test_full[full_feature_cols], test_full["label"]

scale_pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()

clf_xgb_full = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss"
)
clf_xgb_full.fit(X_train_full, y_train_full)

y_pred_xgb_full = clf_xgb_full.predict(X_test_full)
test_full["pred"] = y_pred_xgb_full
test_full["pred_proba"] = clf_xgb_full.predict_proba(X_test_full)[:, 1]

print("=== Tabular + Full Graph (XGBoost) Classification Report ===")
print(classification_report(y_test_full, y_pred_xgb_full, target_names=["licit", "illicit"], digits=4))

xgb_full_overall_f1 = f1_score(y_test_full, y_pred_xgb_full, pos_label=1)
xgb_full_healthy_f1 = f1_score(test_full[test_full.time_step.between(35, 42)]["label"], test_full[test_full.time_step.between(35, 42)]["pred"], pos_label=1)
xgb_full_collapsed_f1 = f1_score(test_full[test_full.time_step >= 43]["label"], test_full[test_full.time_step >= 43]["pred"], pos_label=1)

print(f"Overall Illicit F1 (Full Graph + XGBoost): {xgb_full_overall_f1:.4f}")
print(f"Healthy Period F1 (Steps 35-42):          {xgb_full_healthy_f1:.4f}")
print(f"Collapsed Period F1 (Steps 43-49):        {xgb_full_collapsed_f1:.4f}")


---
# 5. Consistent XGBoost Comparison (Apples-to-Apples Evaluation)

> [!IMPORTANT]
> **Controlled Methodology**: To isolate the exact impact of feature engineering from model architecture, we evaluate all three feature representations using the **exact same XGBoost model configuration** and identical temporal splits:
> 1. **Tabular Only** (165 features)
> 2. **Tabular + Thin Graph** (168 features: tabular + in/out degree + PageRank)
> 3. **Tabular + Full Graph** (500 features: tabular + degree + PageRank + clustering + 2-hop + 165 neighbor means + 165 neighbor maxes)


In [ ]:
# Consistent XGBoost Evaluation Function
def run_consistent_xgb(data_df, feat_list, model_name):
    labeled = data_df[data_df["class"] != "unknown"].copy()
    train_sub = labeled[labeled["time_step"] <= TRAIN_MAX_STEP]
    test_sub = labeled[labeled["time_step"] > TRAIN_MAX_STEP].copy()

    X_tr, y_tr = train_sub[feat_list], train_sub["label"]
    X_te, y_te = test_sub[feat_list], test_sub["label"]

    spw = (y_tr == 0).sum() / (y_tr == 1).sum()

    model = xgb.XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="logloss"
    )
    model.fit(X_tr, y_tr)
    y_pr = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    test_sub["pred"] = y_pr
    test_sub["pred_proba"] = y_proba

    overall_f1 = f1_score(y_te, y_pr, pos_label=1)
    healthy_f1 = f1_score(
        test_sub[test_sub.time_step.between(35, 42)]["label"],
        test_sub[test_sub.time_step.between(35, 42)]["pred"],
        pos_label=1, zero_division=0
    )
    collapsed_f1 = f1_score(
        test_sub[test_sub.time_step >= 43]["label"],
        test_sub[test_sub.time_step >= 43]["pred"],
        pos_label=1, zero_division=0
    )

    print(f"--- {model_name} (Features: {len(feat_list)}) ---")
    print(f"Overall Illicit F1:   {overall_f1:.6f}")
    print(f"Healthy Period (35-42): {healthy_f1:.6f}")
    print(f"Collapsed Period (43-49): {collapsed_f1:.6f}\n")

    return {
        "model_obj": model,
        "test_df": test_sub,
        "summary": {
            "approach": model_name,
            "n_features": len(feat_list),
            "illicit_f1_overall": overall_f1,
            "illicit_f1_healthy_35_42": healthy_f1,
            "illicit_f1_collapsed_43_49": collapsed_f1,
        }
    }

# Run Consistent XGBoost across all three feature sets
res_xgb_tabular = run_consistent_xgb(df, feature_cols, "tabular_only (XGB)")

thin_cols = ["in_degree", "out_degree", "pagerank"]
res_xgb_thin = run_consistent_xgb(df_full_graph, feature_cols + thin_cols, "tabular_plus_thin_graph (XGB)")

res_xgb_full = run_consistent_xgb(df_full_graph, full_feature_cols, "tabular_plus_full_graph (XGB)")

final_xgb_comparison = pd.DataFrame([
    res_xgb_tabular["summary"],
    res_xgb_thin["summary"],
    res_xgb_full["summary"]
])
final_xgb_comparison


In [ ]:
# Visualize Consistent Comparison Bar Chart
x_pos = np.arange(len(final_xgb_comparison))
width = 0.26

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x_pos - width, final_xgb_comparison["illicit_f1_overall"], width, label="Overall Test Period (35-49)", color="#2b5c8f")
ax.bar(x_pos, final_xgb_comparison["illicit_f1_healthy_35_42"], width, label="Healthy Period (35-42)", color="#33a02c")
ax.bar(x_pos + width, final_xgb_comparison["illicit_f1_collapsed_43_49"], width, label="Collapsed Period (43-49)", color="#e31a1c")

ax.set_xticks(x_pos)
ax.set_xticklabels(["Tabular Only", "Tabular + Thin Graph", "Tabular + Full Graph"], fontsize=11)
ax.set_ylabel("Illicit-Class F1 Score", fontsize=11)
ax.set_title("Consistent XGBoost Comparison Across Feature Sets", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(frameon=True)
plt.tight_layout()
plt.savefig("final_comparison_barchart.png", dpi=150, bbox_inches="tight")
plt.show()


### Enhancement: Confusion Matrices (Healthy vs. Collapsed Periods)
We construct confusion matrices for both the **Tabular Baseline (Random Forest)** and the **Consistent Full-Graph XGBoost** model to inspect exact classification counts before and after the shift.


In [ ]:
# Confusion Matrices: Healthy vs Collapsed Periods
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Tabular Baseline (RF) - Healthy Period
cm_rf_healthy = confusion_matrix(
    test_df[test_df.time_step.between(35, 42)]["label"],
    test_df[test_df.time_step.between(35, 42)]["pred_rf"]
)
ConfusionMatrixDisplay(cm_rf_healthy, display_labels=["Licit", "Illicit"]).plot(
    ax=axes[0, 0], cmap="Blues", colorbar=False, values_format="d"
)
axes[0, 0].set_title(f"Tabular Baseline (RF) — Healthy (Steps 35-42)\nF1 = {rf_healthy_f1:.3f}", fontweight="bold")

# 2. Tabular Baseline (RF) - Collapsed Period
cm_rf_collapsed = confusion_matrix(
    test_df[test_df.time_step >= 43]["label"],
    test_df[test_df.time_step >= 43]["pred_rf"]
)
ConfusionMatrixDisplay(cm_rf_collapsed, display_labels=["Licit", "Illicit"]).plot(
    ax=axes[0, 1], cmap="Oranges", colorbar=False, values_format="d"
)
axes[0, 1].set_title(f"Tabular Baseline (RF) — Collapsed (Steps 43-49)\nF1 = {rf_collapsed_f1:.3f}", fontweight="bold")

# 3. Full-Graph XGBoost - Healthy Period
cm_xgb_healthy = confusion_matrix(
    res_xgb_full["test_df"][res_xgb_full["test_df"].time_step.between(35, 42)]["label"],
    res_xgb_full["test_df"][res_xgb_full["test_df"].time_step.between(35, 42)]["pred"]
)
ConfusionMatrixDisplay(cm_xgb_healthy, display_labels=["Licit", "Illicit"]).plot(
    ax=axes[1, 0], cmap="Blues", colorbar=False, values_format="d"
)
axes[1, 0].set_title(f"Full-Graph XGBoost — Healthy (Steps 35-42)\nF1 = {res_xgb_full['summary']['illicit_f1_healthy_35_42']:.3f}", fontweight="bold")

# 4. Full-Graph XGBoost - Collapsed Period
cm_xgb_collapsed = confusion_matrix(
    res_xgb_full["test_df"][res_xgb_full["test_df"].time_step >= 43]["label"],
    res_xgb_full["test_df"][res_xgb_full["test_df"].time_step >= 43]["pred"]
)
ConfusionMatrixDisplay(cm_xgb_collapsed, display_labels=["Licit", "Illicit"]).plot(
    ax=axes[1, 1], cmap="Oranges", colorbar=False, values_format="d"
)
axes[1, 1].set_title(f"Full-Graph XGBoost — Collapsed (Steps 43-49)\nF1 = {res_xgb_full['summary']['illicit_f1_collapsed_43_49']:.3f}", fontweight="bold")

plt.tight_layout()
plt.savefig("confusion_matrices_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


### Enhancement: Split-Boundary Ablation
To prove that the step-43 performance collapse is **intrinsic to the data timeline** (and not an artifact of choosing step 34 as the training boundary), we retrain the tabular model with training cutoffs at **step 30** and **step 38** and compare their illicit F1 over time against the standard **step 34** split.


In [ ]:
# Split-Boundary Ablation Experiment
ablation_results = {}

# Re-use existing step-34 results
ablation_results[34] = [(step, f1_score(group["label"], group["pred_rf"], pos_label=1, zero_division=0)) 
                        for step, group in test_df.groupby("time_step")]

# Train and evaluate cutoff = 30 and cutoff = 38
for cutoff in [30, 38]:
    train_abl = labeled_df[labeled_df["time_step"] <= cutoff]
    test_abl  = labeled_df[labeled_df["time_step"] > cutoff].copy()
    
    clf_abl = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
    clf_abl.fit(train_abl[feature_cols], train_abl["label"])
    
    test_abl["pred"] = clf_abl.predict(test_abl[feature_cols])
    
    f1_list = []
    for step, group in test_abl.groupby("time_step"):
        f1_list.append((step, f1_score(group["label"], group["pred"], pos_label=1, zero_division=0)))
    ablation_results[cutoff] = f1_list

# Plot Ablation Trajectories
plt.figure(figsize=(11, 4.5))
colors = {30: "#1f78b4", 34: "#33a02c", 38: "#e31a1c"}
styles = {30: ":", 34: "-", 38: "--"}

for cutoff, res in ablation_results.items():
    st_vals, f1_vals = zip(*res)
    plt.plot(st_vals, f1_vals, marker="o", label=f"Train $\\leq$ Step {cutoff}", color=colors[cutoff], linestyle=styles[cutoff], lw=2)

plt.axvline(43, color="black", linestyle="--", alpha=0.7, label="Collapse Anchor (Step 43)")
plt.xlabel("Time Step", fontsize=11)
plt.ylabel("Illicit-Class F1 Score", fontsize=11)
plt.title("Split-Boundary Ablation: Illicit F1 Across Training Boundaries (Steps 30, 34, 38)", fontsize=12, fontweight="bold")
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig("split_boundary_ablation.png", dpi=150, bbox_inches="tight")
plt.show()


### Enhancement: SHAP Summary Plot for Best-Performing Model
We compute TreeSHAP values for the **XGBoost (Tabular + Full Graph)** model to explain feature impact. To ensure fast and reliable execution, we evaluate on a stratified subsample of ~2,000 test transactions.


In [ ]:
# SHAP Summary Plot
try:
    import shap
    print("Computing SHAP values for Full-Graph XGBoost model...")
    
    # Subsample 2,000 rows from test set for efficiency
    sample_test = test_full.sample(n=min(2000, len(test_full)), random_state=RANDOM_STATE)
    X_sample = sample_test[full_feature_cols]
    
    explainer = shap.TreeExplainer(clf_xgb_full)
    shap_values = explainer.shap_values(X_sample)
    
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_sample, max_display=15, show=False)
    plt.title("SHAP Feature Summary (XGBoost Full-Graph Model, 2,000 Subsampled Test Rows)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("shap_summary_plot.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"SHAP explanation skipped or encountered notice: {e}")


---
# 6. Distribution Shift Diagnosis

### Why does illicit-class F1 collapse at time step 43?
This section presents the core diagnostic findings:
1. **Sample Size Check**: Verifying that the collapsed period contains sufficient illicit transactions ($N = 22\text{--}36$ per step) to rule out small-sample noise.
2. **Feature Distribution Shift**: Computing the difference in feature means between healthy (steps 35–42) and collapsed (steps 43–49) periods.
3. **Kolmogorov-Smirnov (KS) Statistical Tests**: Proving the shift is statistically significant ($p < 10^{-65}$).
4. **Prediction Confidence Analysis**: Demonstrating that the model becomes **confidently wrong** (median $P(\text{illicit})$ dropping from $0.933 \to 0.040$) rather than merely uncertain.
5. **Licit Class Invariance**: Showing that licit-class F1 remains stable ($0.991 \to 0.987$), confirming the shift is specific to illicit behavior.


In [ ]:
# Shift Diagnosis: Sample Size and Feature Shift Magnitude
healthy_illicit = test_df[(test_df["time_step"].between(35, 42)) & (test_df["label"] == 1)]
collapsed_illicit = test_df[(test_df["time_step"] >= 43) & (test_df["label"] == 1)]

print(f"Illicit transactions in Healthy Period (Steps 35-42):   {len(healthy_illicit):,}")
print(f"Illicit transactions in Collapsed Period (Steps 43-49): {len(collapsed_illicit):,}")

# Feature mean difference between healthy and collapsed periods
healthy_means = healthy_illicit[feature_cols].mean()
collapsed_means = collapsed_illicit[feature_cols].mean()

shift_magnitude = (collapsed_means - healthy_means).abs().sort_values(ascending=False)
top_shift_feats = shift_magnitude.head(4).index.tolist()

print("\nTop 4 Features with Largest Shift Magnitude:")
for feat in top_shift_feats:
    print(f"  - {feat}: Mean Shift = {shift_magnitude[feat]:.4f}")


In [ ]:
# Visualizing the Top Shifted Features as Distributions
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, feat in zip(axes, top_shift_feats):
    ax.hist(healthy_illicit[feat], bins=25, alpha=0.6, label="Healthy (35-42)", density=True, color="#2b5c8f")
    ax.hist(collapsed_illicit[feat], bins=25, alpha=0.6, label="Collapsed (43-49)", density=True, color="#d95f02")
    ax.set_title(f"Feature: {feat}", fontweight="bold")
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("day3_feature_shift_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Kolmogorov-Smirnov Two-Sample Significance Tests
print(f"{'Feature':<12} {'KS Statistic':<15} {'p-value':<15} {'Significance'}")
print("-" * 55)
for feat in top_shift_feats:
    stat, pval = ks_2samp(healthy_illicit[feat], collapsed_illicit[feat])
    sig = "p < 1e-10 (Significant)" if pval < 1e-10 else f"{pval:.2e}"
    print(f"{feat:<12} {stat:<15.4f} {pval:<15.2e} {sig}")


In [ ]:
# Prediction Confidence: Confidently Wrong Analysis
healthy_probs = test_df[(test_df["time_step"].between(35, 42)) & (test_df["label"] == 1)]["pred_proba_rf"]
collapsed_probs = test_df[(test_df["time_step"] >= 43) & (test_df["label"] == 1)]["pred_proba_rf"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(healthy_probs, bins=20, color="#2b5c8f", edgecolor="black", alpha=0.7)
axes[0].axvline(0.5, color="red", linestyle="--", lw=2, label="Decision Threshold (0.5)")
axes[0].set_title(f"Healthy Period (Steps 35-42)\nMedian P(Illicit) = {healthy_probs.median():.3f}", fontweight="bold")
axes[0].set_xlabel("Predicted Probability of Illicit")
axes[0].set_ylabel("Count of True Illicit Txns")
axes[0].legend()

axes[1].hist(collapsed_probs, bins=20, color="#d95f02", edgecolor="black", alpha=0.7)
axes[1].axvline(0.5, color="red", linestyle="--", lw=2, label="Decision Threshold (0.5)")
axes[1].set_title(f"Collapsed Period (Steps 43-49)\nMedian P(Illicit) = {collapsed_probs.median():.3f}", fontweight="bold")
axes[1].set_xlabel("Predicted Probability of Illicit")
axes[1].set_ylabel("Count of True Illicit Txns")
axes[1].legend()

plt.tight_layout()
plt.savefig("day3_prediction_confidence.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Median P(Illicit) for True Illicit in Healthy Period:   {healthy_probs.median():.3f}")
print(f"Median P(Illicit) for True Illicit in Collapsed Period: {collapsed_probs.median():.3f}")


In [ ]:
# Licit Class Stability Check
licit_f1_healthy = f1_score(
    test_df[test_df.time_step.between(35, 42)]["label"],
    test_df[test_df.time_step.between(35, 42)]["pred_rf"],
    pos_label=0
)
licit_f1_collapsed = f1_score(
    test_df[test_df.time_step >= 43]["label"],
    test_df[test_df.time_step >= 43]["pred_rf"],
    pos_label=0
)

print(f"Licit-Class F1 (Healthy Steps 35-42):   {licit_f1_healthy:.4f}")
print(f"Licit-Class F1 (Collapsed Steps 43-49): {licit_f1_collapsed:.4f}")

# Comprehensive Shift Diagnosis Summary Table
shift_summary_table = pd.DataFrame({
    "Diagnostic Metric": [
        "Illicit F1, steps 35-42 (healthy)",
        "Illicit F1, steps 43-49 (collapsed)",
        "Licit F1, steps 35-42 (healthy)",
        "Licit F1, steps 43-49 (collapsed)",
        "Median P(illicit) for true illicit, healthy period",
        "Median P(illicit) for true illicit, collapsed period",
        "Top shifted feature (by mean diff)",
        "KS test p-value for top shifted feature"
    ],
    "Observed Value": [
        f"{rf_healthy_f1:.4f}",
        f"{rf_collapsed_f1:.4f}",
        f"{licit_f1_healthy:.4f}",
        f"{licit_f1_collapsed:.4f}",
        f"{healthy_probs.median():.3f}",
        f"{collapsed_probs.median():.3f}",
        f"{top_shift_feats[0]}",
        f"{ks_2samp(healthy_illicit[top_shift_feats[0]], collapsed_illicit[top_shift_feats[0]])[1]:.2e}"
    ]
})
shift_summary_table


---
# 7. Graph Convolutional Network (GCN with PyTorch Geometric — All 165 Features)

We build a 2-layer Graph Convolutional Network (`GCNConv`) using PyTorch Geometric and evaluate message passing over the entire transaction graph with the exact same temporal boundary (`TRAIN_MAX_STEP = 34`).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

# Set PyTorch random seed for reproducibility
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# Check CUDA / Device availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Computing Device: {device}")

# Map txId -> contiguous index for PyG
txid_to_pyg_idx = {tx: i for i, tx in enumerate(df["txId"])}

x_tensor = torch.tensor(df[feature_cols].values, dtype=torch.float)
y_map = {"1": 1, "2": 0, "unknown": -1}
y_tensor = torch.tensor(df["class"].map(y_map).values, dtype=torch.long)
time_step_tensor = torch.tensor(df["time_step"].values, dtype=torch.long)

edge_src = edges["txId1"].map(txid_to_pyg_idx).values
edge_dst = edges["txId2"].map(txid_to_pyg_idx).values
edge_index = torch.tensor(np.array([edge_src, edge_dst]), dtype=torch.long)

pyg_data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)
pyg_data.time_step = time_step_tensor

labeled_mask = pyg_data.y != -1
pyg_data.train_mask = labeled_mask & (pyg_data.time_step <= TRAIN_MAX_STEP)
pyg_data.test_mask = labeled_mask & (pyg_data.time_step > TRAIN_MAX_STEP)

print(pyg_data)
print(f"Train Masked Nodes: {pyg_data.train_mask.sum().item():,}")
print(f"Test Masked Nodes:  {pyg_data.test_mask.sum().item():,}")


In [ ]:
# Define 2-Layer GCN Architecture
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return x

torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

model_gcn165 = GCN(in_channels=pyg_data.num_features, hidden_channels=64, out_channels=2).to(device)
pyg_data = pyg_data.to(device)
print(model_gcn165)


In [ ]:
# Train GCN (165 Features)
optimizer = torch.optim.Adam(model_gcn165.parameters(), lr=0.01, weight_decay=5e-4)

train_labels = pyg_data.y[pyg_data.train_mask]
n_licit = (train_labels == 0).sum().item()
n_illicit = (train_labels == 1).sum().item()
class_weights = torch.tensor([1.0, n_licit / n_illicit], dtype=torch.float).to(device)

def train_epoch_gcn():
    model_gcn165.train()
    optimizer.zero_grad()
    out = model_gcn165(pyg_data.x, pyg_data.edge_index)
    loss = F.cross_entropy(out[pyg_data.train_mask], pyg_data.y[pyg_data.train_mask], weight=class_weights)
    loss.backward()
    optimizer.step()
    return loss.item()

print("Training GCN (100 Epochs)...")
for epoch in range(1, 101):
    loss_val = train_epoch_gcn()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss_val:.4f}")


In [ ]:
# Evaluate GCN (165 Features)
model_gcn165.eval()
with torch.no_grad():
    out = model_gcn165(pyg_data.x, pyg_data.edge_index)
    pred_gcn = out.argmax(dim=1)

y_true_gcn = pyg_data.y[pyg_data.test_mask].cpu().numpy()
y_pred_gcn = pred_gcn[pyg_data.test_mask].cpu().numpy()

print("=== GCN (All 165 Features) Classification Report ===")
print(classification_report(y_true_gcn, y_pred_gcn, target_names=["licit", "illicit"], digits=4))

gcn_165_overall_f1 = f1_score(y_true_gcn, y_pred_gcn, pos_label=1)
print(f"GCN (All 165 Features) Overall Illicit F1: {gcn_165_overall_f1:.4f}")


In [ ]:
# F1 Over Time for GCN (165 Features)
test_steps_gcn = pyg_data.time_step[pyg_data.test_mask].cpu().numpy()
gcn_results_df = pd.DataFrame({
    "time_step": test_steps_gcn,
    "y_true": y_true_gcn,
    "y_pred": y_pred_gcn
})

f1_per_step_gcn = []
for step, group in gcn_results_df.groupby("time_step"):
    f1 = f1_score(group["y_true"], group["y_pred"], pos_label=1, zero_division=0)
    f1_per_step_gcn.append((step, f1))

steps_gcn_plot, f1s_gcn_plot = zip(*f1_per_step_gcn)

plt.figure(figsize=(10, 4))
plt.plot(steps_gcn_plot, f1s_gcn_plot, marker="o", color="purple", label="GCN (165 Features)")
plt.axvline(43, color="gray", linestyle="--", alpha=0.6, label="Collapse Anchor (Step 43)")
plt.xlabel("Time Step")
plt.ylabel("Illicit-Class F1 Score")
plt.title("GCN (All 165 Features) Illicit F1 Over Time")
plt.legend()
plt.tight_layout()
plt.savefig("day4_gcn_f1_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

gcn_165_healthy_f1 = f1_score(
    gcn_results_df[gcn_results_df.time_step.between(35, 42)]["y_true"],
    gcn_results_df[gcn_results_df.time_step.between(35, 42)]["y_pred"],
    pos_label=1, zero_division=0
)
gcn_165_collapsed_f1 = f1_score(
    gcn_results_df[gcn_results_df.time_step >= 43]["y_true"],
    gcn_results_df[gcn_results_df.time_step >= 43]["y_pred"],
    pos_label=1, zero_division=0
)

print(f"GCN (165 Features) Healthy Period F1:   {gcn_165_healthy_f1:.6f}")
print(f"GCN (165 Features) Collapsed Period F1: {gcn_165_collapsed_f1:.6f}")


---
# 8. Additional Exploration: GCN with 94 Local Features Only

> [!NOTE]
> **Exploration Context**: The Elliptic dataset's 165 features split into 94 local (transaction-specific) and 71 pre-aggregated neighbor features. In the original paper (Weber et al., 2019), GCN models were evaluated on the 94 local features so that message passing would learn neighborhood representations natively.
>
> We replicate this setup using only `feat_1` through `feat_94` to assess whether removing pre-aggregated features affects performance.


In [ ]:
# Build PyG Data with 94 Local Features Only
local_feature_cols = feature_cols[:94]
print(f"Using {len(local_feature_cols)} local features (excluding 71 pre-aggregated features).")

x_local_tensor = torch.tensor(df[local_feature_cols].values, dtype=torch.float)
pyg_local_data = Data(x=x_local_tensor, edge_index=edge_index, y=y_tensor)
pyg_local_data.time_step = time_step_tensor
pyg_local_data.train_mask = labeled_mask & (pyg_local_data.time_step <= TRAIN_MAX_STEP)
pyg_local_data.test_mask = labeled_mask & (pyg_local_data.time_step > TRAIN_MAX_STEP)

torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

model_gcn94 = GCN(in_channels=94, hidden_channels=64, out_channels=2).to(device)
pyg_local_data = pyg_local_data.to(device)

optimizer_local = torch.optim.Adam(model_gcn94.parameters(), lr=0.01, weight_decay=5e-4)

for epoch in range(1, 101):
    model_gcn94.train()
    optimizer_local.zero_grad()
    out = model_gcn94(pyg_local_data.x, pyg_local_data.edge_index)
    loss = F.cross_entropy(out[pyg_local_data.train_mask], pyg_local_data.y[pyg_local_data.train_mask], weight=class_weights)
    loss.backward()
    optimizer_local.step()

# Evaluate GCN (94 Local Features)
model_gcn94.eval()
with torch.no_grad():
    out_local = model_gcn94(pyg_local_data.x, pyg_local_data.edge_index)
    pred_local = out_local.argmax(dim=1)

y_true_local = pyg_local_data.y[pyg_local_data.test_mask].cpu().numpy()
y_pred_local = pred_local[pyg_local_data.test_mask].cpu().numpy()

print("=== GCN (94 Local Features Only) Classification Report ===")
print(classification_report(y_true_local, y_pred_local, target_names=["licit", "illicit"], digits=4))

gcn_local_overall_f1 = f1_score(y_true_local, y_pred_local, pos_label=1)
gcn_results_local_df = pd.DataFrame({
    "time_step": pyg_local_data.time_step[pyg_local_data.test_mask].cpu().numpy(),
    "y_true": y_true_local,
    "y_pred": y_pred_local
})

gcn_local_healthy_f1 = f1_score(
    gcn_results_local_df[gcn_results_local_df.time_step.between(35, 42)]["y_true"],
    gcn_results_local_df[gcn_results_local_df.time_step.between(35, 42)]["y_pred"],
    pos_label=1, zero_division=0
)
gcn_local_collapsed_f1 = f1_score(
    gcn_results_local_df[gcn_results_local_df.time_step >= 43]["y_true"],
    gcn_results_local_df[gcn_results_local_df.time_step >= 43]["y_pred"],
    pos_label=1, zero_division=0
)

print(f"GCN (94 Local) Overall Illicit F1:   {gcn_local_overall_f1:.6f}")
print(f"GCN (94 Local) Healthy Period F1:   {gcn_local_healthy_f1:.6f}")
print(f"GCN (94 Local) Collapsed Period F1: {gcn_local_collapsed_f1:.6f}")


In [ ]:
# Comprehensive 5-Way Comparison Table (Referencing Live Computed Variables)
final_five_way_table = pd.DataFrame([
    {
        "Approach": "Tabular Only (XGBoost)",
        "Features": res_xgb_tabular["summary"]["n_features"],
        "Overall F1": res_xgb_tabular["summary"]["illicit_f1_overall"],
        "Healthy F1 (35-42)": res_xgb_tabular["summary"]["illicit_f1_healthy_35_42"],
        "Collapsed F1 (43-49)": res_xgb_tabular["summary"]["illicit_f1_collapsed_43_49"],
    },
    {
        "Approach": "Tabular + Thin Graph (XGBoost)",
        "Features": res_xgb_thin["summary"]["n_features"],
        "Overall F1": res_xgb_thin["summary"]["illicit_f1_overall"],
        "Healthy F1 (35-42)": res_xgb_thin["summary"]["illicit_f1_healthy_35_42"],
        "Collapsed F1 (43-49)": res_xgb_thin["summary"]["illicit_f1_collapsed_43_49"],
    },
    {
        "Approach": "Tabular + Full Graph (XGBoost)",
        "Features": res_xgb_full["summary"]["n_features"],
        "Overall F1": res_xgb_full["summary"]["illicit_f1_overall"],
        "Healthy F1 (35-42)": res_xgb_full["summary"]["illicit_f1_healthy_35_42"],
        "Collapsed F1 (43-49)": res_xgb_full["summary"]["illicit_f1_collapsed_43_49"],
    },
    {
        "Approach": "GCN (All 165 Features)",
        "Features": pyg_data.num_features,
        "Overall F1": gcn_165_overall_f1,
        "Healthy F1 (35-42)": gcn_165_healthy_f1,
        "Collapsed F1 (43-49)": gcn_165_collapsed_f1,
    },
    {
        "Approach": "GCN (94 Local Features Only)",
        "Features": len(local_feature_cols),
        "Overall F1": gcn_local_overall_f1,
        "Healthy F1 (35-42)": gcn_local_healthy_f1,
        "Collapsed F1 (43-49)": gcn_local_collapsed_f1,
    },
])
final_five_way_table.to_csv("final_five_way_comparison.csv", index=False)
final_five_way_table


---
# 9. Literature Comparison (Weber et al., KDD 2019)

### Context & Methodology Comparison
The Elliptic dataset was published alongside **Weber et al. (2019)**: *"Anti-Money Laundering in Bitcoin: Experimenting with Graph Convolutional Networks for Financial Forensics"* (ACM SIGKDD 2019 Workshop on Applied Data Science).

| Benchmark / Model | Weber et al. (2019) Reported Illicit F1 | Our Merged Project Illicit F1 | Key Architectural & Experimental Differences |
|---|---|---|---|
| **Random Forest (Tabular)** | ~0.78 – 0.80 | **0.812** (Overall) / **0.896** (Healthy) | Weber et al. evaluated across sliding test splits; our single split (1–34 train, 35–49 test) isolates the post-step-42 collapse. |
| **XGBoost (Tabular)** | ~0.81 (Static baseline) | **0.784** (Overall) / **0.892** (Healthy) | Consistent tree depth ($6$), learning rate ($0.05$), and `scale_pos_weight` without excessive tuning. |
| **XGBoost (Full Graph)** | *N/A (Did not engineer 2-hop/clustering)* | **0.808** (Overall) / **0.906** (Healthy) | Novel engineering: 2-hop reach, undirected clustering coefficient, and full 165-feature mean/max aggregations. |
| **GCN (94 Local Features)** | ~0.50 (Static GCN) | **0.297** (Overall) / **0.436** (Healthy) | Weber et al. utilized dynamic EvolveGCN architectures with extensive learning rate schedules; our minimal 2-layer GCN lacked temporal recurrence. |
| **GCN (All 165 Features)** | ~0.74 (Dynamic EvolveGCN) | **0.499** (Overall) / **0.601** (Healthy) | Pre-aggregated neighbor features significantly boost static GCN performance ($0.297 \to 0.499$). |

> [!NOTE]
> **Setup Differences & Caveats**:
> 1. **Temporal Averaging**: Aggregate F1 scores reported in literature often average over time windows that blend healthy and post-shift periods or utilize expanding training windows.
> 2. **Model Complexity**: High-performing GNNs in the literature rely on dynamic recurrent architectures (such as EvolveGCN-O / EvolveGCN-H) that continuously adapt node weights over time, whereas our GCN is a static 2-layer spatial baseline.


---
# 10. Discussion: Addressing Distribution Shift in Production AML

While solving the distribution shift was outside the scope of this baseline study, our empirical findings point directly to critical design requirements for production cryptocurrency transaction monitoring systems:

### 1. Continuous Drift Detection & Statistical Alarming
- **Feature-Level Drift**: Real-time statistical tests (such as the two-sample **Kolmogorov-Smirnov test** or **Population Stability Index (PSI)**) should continuously monitor the incoming distribution of high-shift features (e.g., `feat_114`, `feat_53`, `feat_54`).
- **Prediction Confidence Shifts**: In our analysis, median prediction confidence for true illicit entities collapsed from $0.933 \to 0.040$. Tracking rolling confidence distributions and monitoring the output entropy of the classifier acts as an immediate early-warning signal that the underlying data-generating distribution has shifted.

### 2. Adaptive & Sliding-Window Retraining Cadences
- Static models trained once on historical patterns are actively hazardous in adversarial domains: illicit actors actively adapt techniques to evade detection.
- Implementing an automated, time-based sliding window retraining cadence (e.g., updating models every 2–4 time steps with newly confirmed investigative labels) prevents the model from remaining locked to obsolete behavioral patterns.

### 3. Semi-Supervised & Self-Supervised Graph Learning
- In this dataset, **77% of all transactions are labeled 'unknown'**.
- Production systems can leverage these unlabeled nodes via **transductive graph neural networks**, **Contrastive Graph Learning**, or **Graph Autoencoders** to capture topological shifts and entity clustering dynamics long before explicit investigative ground-truth labels become available.
